# 04 - Batch/Cohort Baseline vs Context-Enriched Comparison

This notebook compares two cohort-level conditions:
1) Baseline prompt strategies (no per-student mental model)
2) Context-enriched prompt strategies (same base strategy + per-student mental model context)

The goal is to measure whether per-student mental model context improves quality signals at scale.

In [ ]:
import json
import time
import pandas as pd
from google.genai import types

from lib.experiment_utils import (
    create_client,
    load_best_attempts_df,
    select_cohort_ids,
    get_representative_rows_per_student,
    build_strategies,
    save_results,
)
from lib.llm_batch_analyzer import format_submissions, clean_json_response
from lib.mental_model import (
    load_skill_map,
    calculate_student_profile,
    build_prerequisite_graph,
    get_weak_skills,
    build_mental_model_payload,
)

MODEL_ID = "gemini-2.5-flash"
RANDOM_SEED = 42
BATCH_SIZE = 50
MIN_SUBMISSIONS = 5
SLEEP_SECONDS = 1.0

client = create_client()
print(f"Ready. Model: {MODEL_ID}")

In [ ]:
best_attempts_df = load_best_attempts_df()
cohort_ids = select_cohort_ids(
    best_attempts_df=best_attempts_df,
    batch_size=BATCH_SIZE,
    min_submissions=MIN_SUBMISSIONS,
    seed=RANDOM_SEED,
)

if not cohort_ids:
    raise ValueError("No eligible students found for cohort experiment.")

cohort_df = best_attempts_df[best_attempts_df["SubjectID"].isin(cohort_ids)].copy()
rep_df = get_representative_rows_per_student(cohort_df, cohort_ids)

print(f"Cohort size: {len(cohort_ids)}")
print(f"Representative submissions: {len(rep_df)}")
display(rep_df[["SubjectID", "ProblemID", "Score"]].head())

In [ ]:
strategies = build_strategies(focus_problem_ids=list(cohort_df["ProblemID"].unique()))

skill_map, all_skills = load_skill_map()
graph = build_prerequisite_graph()

mental_model_by_student = {}
for sid in cohort_ids:
    profile = calculate_student_profile(sid, best_attempts_df, skill_map, all_skills)
    weak_skills = get_weak_skills(profile, threshold=0.6)
    mental_model_by_student[sid] = build_mental_model_payload(
        student_id=sid,
        profile=profile,
        weak_skill_pairs=weak_skills,
        graph=graph,
    )

print(f"Built mental models for {len(mental_model_by_student)} students")

In [ ]:
def add_mental_model_context(base_prompt: str, mental_model: dict) -> str:
    context = json.dumps(mental_model, indent=2)
    return (
        base_prompt
        + "\n\nAdditional Student Mental Model Context:\n"
        + context
        + "\n\nUse this context to better judge likely misconceptions and future risks."
    )

def run_one_call(submission_row, system_instruction: str):
    formatted_input = format_submissions([submission_row.to_dict()])
    t0 = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
                response_mime_type="application/json",
            ),
        )
        raw_text = response.text if response and response.text else "{}"
        parsed = json.loads(clean_json_response(raw_text))
        return parsed, round(time.time() - t0, 3), None
    except Exception as e:
        return None, round(time.time() - t0, 3), str(e)

def valid_output(value):
    return not (isinstance(value, str) and value.startswith("Error:"))

In [ ]:
rows = []

for _, sub in rep_df.iterrows():
    sid = sub["SubjectID"]
    mm = mental_model_by_student.get(sid, {})

    for strategy_name, prompt_fn in strategies.items():
        baseline_prompt = prompt_fn()
        enriched_prompt = add_mental_model_context(baseline_prompt, mm)

        baseline_out, baseline_time, baseline_err = run_one_call(sub, baseline_prompt)
        enriched_out, enriched_time, enriched_err = run_one_call(sub, enriched_prompt)

        rows.append({
            "SubjectID": sid,
            "ProblemID": sub["ProblemID"],
            "Score": sub["Score"],
            "Strategy": strategy_name,
            "Baseline_Output": baseline_out if baseline_err is None else f"Error: {baseline_err}",
            "Baseline_TimeSec": baseline_time,
            "ContextEnriched_Output": enriched_out if enriched_err is None else f"Error: {enriched_err}",
            "ContextEnriched_TimeSec": enriched_time,
        })

        if SLEEP_SECONDS > 0:
            time.sleep(SLEEP_SECONDS)

comparison_df = pd.DataFrame(rows)
print(f"Rows generated: {len(comparison_df)}")
display(comparison_df.head())

In [ ]:
summary_df = comparison_df.groupby("Strategy").apply(
    lambda g: pd.Series({
        "Rows": len(g),
        "Students": int(g["SubjectID"].nunique()),
        "Baseline_ValidPct": round(100 * g["Baseline_Output"].apply(valid_output).mean(), 1),
        "ContextEnriched_ValidPct": round(100 * g["ContextEnriched_Output"].apply(valid_output).mean(), 1),
        "Baseline_AvgTime": round(g["Baseline_TimeSec"].mean(), 3),
        "ContextEnriched_AvgTime": round(g["ContextEnriched_TimeSec"].mean(), 3),
        "Delta_Time": round(g["ContextEnriched_TimeSec"].mean() - g["Baseline_TimeSec"].mean(), 3),
    })
).reset_index()

display(summary_df)

raw_csv = f"batch_{len(rep_df)}_baseline_vs_context_enriched_raw.csv"
summary_csv = f"batch_{len(rep_df)}_baseline_vs_context_enriched_summary.csv"
save_results(comparison_df, raw_csv)
save_results(summary_df, summary_csv)
print(f"Saved: {raw_csv}")
print(f"Saved: {summary_csv}")